## Extracting Drug Allergies For Each Patient
This notebook takes the full MIMIC_IV_EXT_BHC dataset, and performs the following 
1. On the 'text' column: drop unused sections of text. Sections are prefixed by \<SECTION HEADERS\>. These unused sections are either represented elsewhere, rare (some have only 1 instance) or should not be sections.
2. Creates a dataframe of note_id, section, text, and saves that for future use
3. From this dataframe, extract out only the ALLERGIES section, and combines this with the main MIMIC_IV patient dataset (the EXT_BHC dataset has only note_id to identify patient) to get a new dataframe of subject_id, drugs, and saves that for further processing in another notebook.

The MIMIC_IV_EXT_BHC dataset is available to credentialled MIMIC-IV users at https://physionet.org/content/labelled-notes-hospital-course/1.0.0/. Here I have renamed my download to bhc.csv for easier typing.

Citation:  
Johnson, A., Pollard, T., Horng, S., Celi, L. A., & Mark, R. (2023). MIMIC-IV-Note: Deidentified free-text clinical notes (version 2.2). PhysioNet. RRID:SCR_007345. https://doi.org/10.13026/1n74-ne17  
Goldberger, A., Amaral, L., Glass, L., Hausdorff, J., Ivanov, P. C., Mark, R., ... & Stanley, H. E. (2000). PhysioBank, PhysioToolkit, and PhysioNet: Components of a new research resource for complex physiologic signals. Circulation [Online]. 101 (23), pp. e215–e220. RRID:SCR_007345.


In [ ]:
import pandas as pd
data = pd.read_csv('bhc.csv')

# ---------------------------------------------------------------------------
# 1.  CONFIGURATION
# ---------------------------------------------------------------------------

# Sections dropped unconditionally — already captured previously.
SECTIONS_TO_DROP: set[str] = {"SEX", "SERVICE", "ATTENDING"}

# Default section set for topic modelling.
# Chosen for semantic richness; excludes sections that are:
#   - Pure boilerplate  (DISCHARGE INSTRUCTIONS, FOLLOWUP INSTRUCTIONS)
#   - Single-word values (DISCHARGE DISPOSITION → "Home" / "SNF")
#   - Mostly numeric    (PERTINENT RESULTS → almost entirely lab values)
#   - Structured lists  (MEDICATIONS ON ADMISSION, DISCHARGE MEDICATIONS)
# Adjust freely — use discover_sections() to inform your choices.

SECTIONS_TO_DROP: set[str] = {
# these should not be section headers at all
'ALBUMIN',
'ATTENDING',
'BR',
'COLLECT',
'DC EXAM',
'DIFFICULT INTUBATION DURING HER TONSILLECTOMY',
'FINAL REPORT IS PENDING',
'FINAL REPORT IS PENDING AT TIME OF DISCHARGE',
'IMAGING',
'INSERT',
'MICROBIOLOGY',
'ON ADMISSION',
'PATHOLOGY',
'SERVICE',
'SEX'}

DEFAULT_SECTIONS: set[str] = {
    "CHIEF COMPLAINT",
    "HISTORY OF PRESENT ILLNESS",
    "PAST MEDICAL HISTORY",
    "SOCIAL HISTORY",
    "FAMILY HISTORY",
    "PHYSICAL EXAM",
    "DISCHARGE DIAGNOSIS",
    "DISCHARGE CONDITION",
    "MAJOR SURGICAL OR INVASIVE PROCEDURE",
    "ALLERGIES",
    "DISCHARGE INSTRUCTIONS",
    "PERTINENT RESULTS",
    "MEDICATIONS ON ADMISSION",
    "DISCHARGE DISPOSITION",
    "FOLLOWUP INSTRUCTIONS",
    "DISCHARGE MEDICATIONS"    
}

In [ ]:
# ---------------------------------------------------------------------------
# 2.  SECTION PARSER
# ---------------------------------------------------------------------------
import re

SECTION_TAG_RE = re.compile(r"<([A-Z][A-Z0-9\s/()]+?)>")

def parse_sections(raw_text: str) -> dict[str, str]:
    """
    Split raw note on <SECTION> tags.
    Returns {section_name: raw_text} for all non-admin sections.
    """
    sections: dict[str, str] = {}
    matches = list(SECTION_TAG_RE.finditer(raw_text))

    if not matches:
        return {"FULL_TEXT": raw_text.strip()}

    for i, match in enumerate(matches):
        name = match.group(1).strip()
        if name in SECTIONS_TO_DROP:
            continue
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(raw_text)
        sections[name] = raw_text[start:end].strip()

    return sections


In [ ]:
# ---------------------------------------------------------------------------
# 3.  TEXT CLEANERS
# ---------------------------------------------------------------------------

# MIMIC de-id placeholder
_DEID        = re.compile(r"\b___+\b")

# Timestamps: 10:25PM, 10:25:00, and residual fragments like ": 07PM"
_TIMESTAMP   = re.compile(
    r"(?:^|(?<=\s))\d{1,2}:\d{2}(?::\d{2})?(?:\s?[APap][Mm])?\b"
    r"|:\s*\d{2}[APap][Mm]\b"
)

# Lab key-value pairs: SODIUM-138, WBC-5.0#, GLUCOSE-109*
_LAB_KV      = re.compile(r"\b[A-Z][A-Z0-9\s\(\)/]+[-\u2013]\d+\.?\d*[*#]?\b")

# Residual lab suffixes left after _LAB_KV removal
_LAB_SUFFIX  = re.compile(r"(?<!\w)[*#]+(?!\w)")

# Bare numbers with no clinical unit attached.
# Keeps: "100 mcg", "40 mg", "BMI of 45 kg" etc.
_BARE_NUM    = re.compile(
    r"(?<!\w)\d+\.?\d*[*#%]?"
    r"(?!\s*(?:mg|mcg|mEq|mL|L|kg|g|lb|mmHg|bpm|cm|mm|%|"
    r"days?|weeks?|months?|years?|hours?|tabs?|capsules?|units?|u\b|iu\b))"
    r"(?!\w)"
)

# Orphaned list markers left after number removal: "1." "2." ". "
_LIST_MARKER = re.compile(r"(?:^|\s)\d*\.\s+")

# Multi-char punctuation runs (but not single "." — preserve sentence endings)
_PUNCT_NOISE = re.compile(r"[-\u2013_:,;]{2,}|\.{2,}")

# Whitespace normaliser
_WHITESPACE  = re.compile(r"\s+")

# Segments that are too short to be meaningful after cleaning
_BARE_PERIOD = re.compile(r"^\.*\s*$")


def clean_section_text(text: str) -> str:
    """
    Clean a single section's raw text.
    Returns the cleaned string, or "" if nothing meaningful remains.
    Empty returns are silently dropped upstream — never passed to BERTopic.
    """
    if not text or not text.strip():
        return ""

    t = text
    t = _DEID.sub(" ", t)
    t = _TIMESTAMP.sub(" ", t)
    t = _LAB_KV.sub(" ", t)
    t = _BARE_NUM.sub(" ", t)
    t = _LAB_SUFFIX.sub(" ", t)
    t = _LIST_MARKER.sub(" ", t)
    t = _PUNCT_NOISE.sub(" ", t)
    t = _WHITESPACE.sub(" ", t).strip()

    # Discard if only punctuation / whitespace remains, or a null-meaning word
    _NULL_WORDS = {"none", "n/a", "na", "nil", "unknown", "not applicable"}
    if _BARE_PERIOD.match(t) or len(t.replace(" ", "").replace(".", "")) < 3:
        return ""
    if t.strip().lower() in _NULL_WORDS:
        return ""

    return t


In [ ]:
# ---------------------------------------------------------------------------
# 4.  SINGLE NOTE PROCESSOR
# ---------------------------------------------------------------------------

def preprocess_note(
    note_id: str,
    raw_text: str,
    sections: set[str] | None = None,
) -> list[dict]:
    """
    Preprocess one note. Returns one dict per non-empty, included section:

        {
            "note_id": str,
            "section": str,   # e.g. "HISTORY OF PRESENT ILLNESS"
            "text":    str,   # cleaned, non-empty text
        }

    Segments that clean to "" are silently discarded.

    Args:
        note_id:  Identifier for this note.
        raw_text: Raw input string from the dataset.
        sections: Set of section names to keep. Defaults to DEFAULT_SECTIONS.
                  Pass None to keep all non-admin, non-empty sections.
    """
    include = sections if sections is not None else DEFAULT_SECTIONS
    parsed = parse_sections(raw_text)
    records = []

    for section_name, section_text in parsed.items():
        # Section filter: skip if not in the include set (unless include is empty = keep all)
        if include and section_name not in include:
            continue

        cleaned = clean_section_text(section_text)
        if not cleaned:          # silently drop empty segments
            continue

        records.append({
            "note_id": note_id,
            "section": section_name,
            "text":    cleaned,
        })

    return records

In [ ]:
# ---------------------------------------------------------------------------
# 5.  DATASET-LEVEL PROCESSOR
# ---------------------------------------------------------------------------

def preprocess_dataset(
    df: pd.DataFrame,
    note_id_col: str = "note_id",
    text_col: str = "input",
    sections: set[str] | None = None,
) -> tuple[list[str], list[dict]]:
    """
    Preprocess all notes in a DataFrame for BERTopic.

    Args:
        df:           DataFrame with note_id and input columns.
        note_id_col:  Name of the note ID column.
        text_col:     Name of the raw text column.
        sections:     Set of section names to keep.
                      Defaults to DEFAULT_SECTIONS (the 10 semantically rich
                      sections identified from the 1000-note sample).
                      Pass an explicit set to override, e.g.:
                          sections={"HISTORY OF PRESENT ILLNESS",
                                    "PAST MEDICAL HISTORY"}
                      Pass set() (empty set) to keep ALL non-admin sections
                      that have content after cleaning.

    Returns:
        docs:  List[str]  — cleaned text strings; no None, no empty strings.
        meta:  List[dict] — parallel metadata list, same length as docs.
                            Each dict has: note_id, section.
                            Join topic results back via note_id.
    """
    # Treat empty set as "keep everything"
    include = sections if sections is not None else DEFAULT_SECTIONS
    all_docs: list[str] = []
    all_meta: list[dict] = []

    for _, row in df.iterrows():
        note_id  = str(row[note_id_col])
        raw_text = str(row[text_col])

        for seg in preprocess_note(note_id, raw_text, sections=include):
            all_docs.append(seg["text"])
            all_meta.append({"note_id": seg["note_id"], "section": seg["section"]})

    return all_docs, all_meta

In [ ]:
# ---------------------------------------------------------------------------
# 6.  SECTION DISCOVERY UTILITY
# ---------------------------------------------------------------------------

def discover_sections(
    df: pd.DataFrame,
    text_col: str = "input",
    min_count: int = 1,
) -> pd.DataFrame:
    """
    Scan every note and report all unique <SECTION> tags with frequency stats.

    Returns a DataFrame with columns:
        section       — tag name
        count         — number of notes containing this section
        pct_of_notes  — % of notes
        in_default    — True if in DEFAULT_SECTIONS (will be kept by default)
        is_dropped    — True if in SECTIONS_TO_DROP (admin, always excluded)

    Use this before preprocessing to review coverage and refine your
    section selection.
    """
    counter: Counter = Counter()
    n_notes = len(df)

    for raw_text in df[text_col].astype(str):
        found = {m.group(1).strip() for m in SECTION_TAG_RE.finditer(raw_text)}
        counter.update(found)

    rows = [
        {
            "section":      section,
            "count":        count,
            "pct_of_notes": round(100 * count / n_notes, 1),
            "in_default":   section in DEFAULT_SECTIONS,
            "is_dropped":   section in SECTIONS_TO_DROP,
        }
        for section, count in counter.most_common()
        if count >= min_count
    ]
    return pd.DataFrame(rows)

In [ ]:
# ---------------------------------------------------------------------------
# 7.  STORAGE HELPER
# ---------------------------------------------------------------------------

def segments_to_dataframe(docs: list[str], meta: list[dict]) -> pd.DataFrame:
    """
    Combine the parallel docs + meta lists into a tidy DataFrame.

    Columns: note_id, section, text

    Typical workflow:
        docs, meta = preprocess_dataset(df)
        seg_df = segments_to_dataframe(docs, meta)
        seg_df.to_parquet("mimic_segments.parquet", index=False)

        # After BERTopic:
        seg_df["topic"]       = topics
        seg_df["topic_prob"]  = probs
        seg_df["topic_label"] = topic_model.get_topic_info()...
    """
    return pd.DataFrame([
        {"note_id": m["note_id"], "section": m["section"], "text": text}
        for text, m in zip(docs, meta)
    ])


In [ ]:
# ── 1. Section discovery ─────────────────────────────────────────────────
from collections import Counter

print("\n" + "=" * 65)
print("  SECTION DISCOVERY")
print("=" * 65)
sec_df = discover_sections(data)
print(sec_df.to_string(index=False))

In [ ]:
# ── 2. Default preprocessing ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  PREPROCESSING  (default section set)")
print("=" * 65)
docs, meta = preprocess_dataset(data)
print(f"  Notes processed          : {len(data)}")
print(f"  Segments kept            : {len(docs)}")
print(f"  Avg segments / note      : {len(docs)/len(data):.1f}")
print(f"  Any None?                : {any(d is None for d in docs)}")
print(f"  Any empty string?        : {any(d == '' for d in docs)}")
sections_found = {m['section'] for m in meta}
print(f"  Sections present in output: {sorted(sections_found)}")

In [ ]:
# ── 4. Segments DataFrame ────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  SEGMENTS DATAFRAME  (first 10 rows)")
print("=" * 65)
seg_df = segments_to_dataframe(docs, meta)
print(seg_df.head(10).to_string(index=False, max_colwidth=80))
print(f"\n  Full shape: {seg_df.shape}")

In [ ]:
seg_df.to_csv('MIMIC_IV_EXT_BHC_Full_notes_by_section.csv', index=False)

In [ ]:
# Slice out the ALLERGIES section only and remove non-drug entries
allergies = seg_df[seg_df['section']=='ALLERGIES']
allergies = allergies[allergies['text'] != 'No Known Allergies / Adverse Drug Reactions']
allergies = allergies[allergies['text'] != 'Patient recorded as having No Known Allergies to Drugs']
allergies

In [ ]:
# merge with main MIMIC_IV dataset, the common attribute is note_id
# drop NA - these are patients without allergies.

patients = pd.read_csv('discharge_1.csv'  # path to MIMIC-IV-Note discharge file)
patients = patients.drop(['text', 'hadm_id', 'charttime', 'storetime'], axis=1)
patient_allergies = pd.merge(patients, allergies[['note_id', 'text']], on='note_id', how='left')
patient_allergies_clean = patient_allergies.dropna(subset = ['text'])

In [ ]:
def extract_drugs(text):
    # Split on common delimiters: '/', newline, comma, semicolon
    drugs = re.split(r'[/,;\n]+', text)
    # Strip whitespace and punctuation, lowercase for dedup, drop empty strings
    drugs = [d.strip().strip('.,;:()') for d in drugs]
    drugs = [d for d in drugs if d]
    return drugs

In [ ]:
# Extract unique drug names per subject (case-insensitive dedup)
def unique_drugs(text):
    drugs = extract_drugs(text)
    seen = set()
    unique = []
    for d in drugs:
        key = d.lower()
        if key not in seen:
            seen.add(key)
            unique.append(d)
    return unique

In [ ]:
# Group by subject_id, concatenate all text
grouped = (
    patient_allergies_clean.groupby('subject_id')['text']
    .apply(lambda texts: '/'.join(texts.dropna().astype(str)))  # join all rows' text
    .reset_index()
)

In [ ]:
# explode the drugs so that one row = one drug entry
grouped['drugs'] = grouped['text'].apply(unique_drugs)
grouped = grouped.drop(columns='text')
exploded = grouped.explode('drugs').reset_index(drop=True)


In [ ]:
# combine all the contrasts entries into 1 entry
# remove all the other non-drug entries.

CONTRAST_VARIANTS = {
    'contrast', 'contrast dye', 'iodine contrast', 'iodinated contrast',
    'contrast media', 'iv contrast', 'oral contrast', 'contrast agent'
}

NON_DRUG_TERMS = {
    'nka', 'nkda', 'no known allergies', 'no known drug allergies',
    'latex', 'tape', 'adhesive', 'shellfish', 'bee sting', 'unknown'
}

def pre_filter(df):
    """
    - Normalise contrast variants to the string 'contrast'
    - Drop rows that are non-drug terms
    Applied to the exploded dataframe.
    """
    # Lowercase version for matching only
    lowered = df['drugs'].str.strip().str.lower()

    # Replace contrast variants with canonical 'contrast'
    df = df.copy()
    df.loc[lowered.isin(CONTRAST_VARIANTS), 'drugs'] = 'contrast'

    # Drop non-drug terms entirely
    df = df[~lowered.isin(NON_DRUG_TERMS)].reset_index(drop=True)

    return df

In [ ]:
exploded_clean = pre_filter(exploded)
exploded_clean

In [ ]:
exploded_clean.to_csv('MIMIC_IV_PATIENT_DRUG_ALLERGIES.csv', index=False)